In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260531_082518"

snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots.parquet"))

# RAW DATA
#    ↓
# REGIME FEATURES (no future)
#    ↓
# FIT SCALER + GMM
#    ↓
# ASSIGN REGIMES
#    ↓
# BUILD FUTURE LABELS (separate)
#    ↓
# JOIN ON TIME INDEX
#    ↓
# REGIME STATISTICS / LABELING

In [2]:
# STEP 1 — Load raw data

df = snapshots
df = df.sort_values("ts").reset_index(drop=True)

df["ts"] = pd.to_datetime(df["ts"])
df = df.set_index("ts")

In [10]:
df

,symbol,best_bid,best_ask,mid,microprice,best_bid_tick,best_ask_tick,mid_tick,spread,order_imbalance,...,my_bid_tick,my_ask_tick,bid_distance_touch,ask_distance_touch,bid_distance_spread,ask_distance_spread,bid_delta,ask_delta,quote_churn,future_return
ts,,,,,,,,,,,,,,,,,,,,,
1970-01-01 00:29:40.199666414,BTCUSDT,74165.66,74165.67,74165.665,74165.660604,7416566,7416567,7416567,0.01,-0.875907,...,7416560,7416567,-0.06,0.00,-0.07,0.01,0.00,0.0,0.00,0.0
1970-01-01 00:29:40.199666515,BTCUSDT,74165.66,74165.67,74165.665,74165.660604,7416566,7416567,7416567,0.01,-0.875907,...,7416560,7416567,-0.06,0.00,-0.07,0.01,0.00,0.0,0.00,0.0
1970-01-01 00:29:40.199666616,BTCUSDT,74165.66,74165.67,74165.665,74165.660605,7416566,7416567,7416567,0.01,-0.875744,...,7416560,7416567,-0.06,0.00,-0.07,0.01,0.00,0.0,0.00,0.0
1970-01-01 00:29:40.199666714,BTCUSDT,74165.66,74165.67,74165.665,74165.660622,7416566,7416567,7416567,0.01,-0.872296,...,7416560,7416567,-0.06,0.00,-0.07,0.01,0.00,0.0,0.00,0.0
1970-01-01 00:29:40.199666814,BTCUSDT,74165.66,74165.67,74165.665,74165.660622,7416566,7416567,7416567,0.01,-0.872296,...,7416560,7416567,-0.06,0.00,-0.07,0.01,0.00,0.0,0.00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1970-01-01 00:29:40.215917114,BTCUSDT,73975.62,73975.63,73975.625,73975.629194,7397562,7397563,7397562,0.01,0.842200,...,7397562,7397571,0.00,0.08,-0.01,0.09,0.01,0.0,0.01,NaN
1970-01-01 00:29:40.215917214,BTCUSDT,73975.62,73975.63,73975.625,73975.629195,7397562,7397563,7397562,0.01,0.842297,...,7397562,7397571,0.00,0.08,-0.01,0.09,0.01,0.0,0.01,NaN
1970-01-01 00:29:40.215917314,BTCUSDT,73975.62,73975.63,73975.625,73975.629195,7397562,7397563,7397562,0.01,0.842297,...,7397562,7397571,0.00,0.08,-0.01,0.09,0.01,0.0,0.01,NaN


In [ ]:
# STEP 2 — Build REGIME FEATURES (ONLY past info) slower trends - 1000ms

"""
2. Choose regime window (critical design choice)
Start simple:
"""

WINDOW = "1s"   # later try 2s, 5s

regime_df = pd.DataFrame()

regime_df["volatility"] = df["mid"].pct_change().rolling(WINDOW).std()
regime_df["spread"] = df["spread"].rolling(WINDOW).mean()
regime_df["order_imbalance"] = df["order_imbalance"].rolling(WINDOW).mean()
regime_df["trade_imbalance"] = df["trade_imbalance"].rolling(WINDOW).mean()
regime_df["quote_churn"] = df["quote_churn"].rolling(WINDOW).mean()
regime_df["inventory"] = df["inventory"].rolling(WINDOW).mean()
regime_df["inventory_vol"] = df["inventory"].rolling(WINDOW).std()
regime_df["microprice_error"] = (df["mid"] - df["microprice"]).rolling(WINDOW).mean()

regime_df = regime_df.dropna()

In [5]:
regime_df

,volatility,spread,order_imbalance,trade_imbalance,quote_churn,inventory,inventory_vol,microprice_error
ts,,,,,,,,
1970-01-01 00:29:40.199666616,0.000000,0.01000,-0.875853,0.148800,0.000000,0.000000,0.000000,0.004396
1970-01-01 00:29:40.199666714,0.000000,0.01000,-0.874964,0.179200,0.000000,0.000000,0.000000,0.004391
1970-01-01 00:29:40.199666814,0.000000,0.01000,-0.874430,0.197440,0.000000,0.000000,0.000000,0.004388
1970-01-01 00:29:40.199666914,0.000000,0.01000,-0.873962,0.133376,0.001667,0.000000,0.000000,0.004386
1970-01-01 00:29:40.199667016,0.000000,0.01000,-0.873699,0.064386,0.001429,0.000000,0.000000,0.004385
...,...,...,...,...,...,...,...,...
1970-01-01 00:29:40.215917114,0.000005,0.01006,-0.080021,-0.009436,0.049374,0.674735,0.993672,0.000418
1970-01-01 00:29:40.215917214,0.000005,0.01006,-0.080015,-0.009430,0.049374,0.674724,0.993678,0.000418
1970-01-01 00:29:40.215917314,0.000005,0.01006,-0.080009,-0.009424,0.049374,0.674714,0.993684,0.000418


In [ ]:
# STEP 3 — Train regime model

scaler = StandardScaler()
X = scaler.fit_transform(regime_df) # train with pure np array not df

n_regimes = 4  # start small: 2–5 max

model = GaussianMixture(n_components=n_regimes, covariance_type="full", random_state=42)

regime_df["regime"] = model.fit_predict(X)

In [7]:
# STEP 4 — NOW build evaluation labels (separate dataset)

eval_df = df.copy()
mid = eval_df["mid"].values
h = 10 # 1000ms window

future_return = np.full(len(df), np.nan)  # creates an array the same length as your dataset and fills every element with NaN 
future_volatility = np.full(len(df), np.nan)
future_direction = np.full(len(df), np.nan)

for i in range(len(df) - h):

    p0 = mid[i]
    p1 = mid[i + h]

    window = mid[i:i + h]

    # 1. Return (trend / drift)
    future_return[i] = (p1 - p0) / p0

    # 2. Realized volatility in future window
    future_volatility[i] = np.std(np.diff(window) / window[:-1])

    # 3. Direction (simple sign regime)
    future_direction[i] = np.sign(p1 - p0)

# 👇 INSERT HERE (this is the key step)
eval_df["future_return"] = future_return
eval_df["future_volatility"] = future_volatility
eval_df["future_direction"] = future_direction 

# Future direction regime
# If result ≈ +1
# almost always up moves after this regime
# strong bullish bias
# If result ≈ -1
# almost always down moves after this regime
# bearish bias
# If result ≈ 0
# no directional bias
# pure noise / mean reversion / stable

# optional cleanup AFTER labeling
eval_df = eval_df.dropna(subset=["future_return", "future_volatility", "future_direction"])

In [8]:
eval_df

,symbol,best_bid,best_ask,mid,microprice,best_bid_tick,best_ask_tick,mid_tick,spread,order_imbalance,...,bid_distance_touch,ask_distance_touch,bid_distance_spread,ask_distance_spread,bid_delta,ask_delta,quote_churn,future_return,future_volatility,future_direction
ts,,,,,,,,,,,,,,,,,,,,,
1970-01-01 00:29:40.199666414,BTCUSDT,74165.66,74165.67,74165.665,74165.660604,7416566,7416567,7416567,0.01,-0.875907,...,-0.06,0.00,-0.07,0.01,0.00,0.0,0.00,0.0,0.0,0.0
1970-01-01 00:29:40.199666515,BTCUSDT,74165.66,74165.67,74165.665,74165.660604,7416566,7416567,7416567,0.01,-0.875907,...,-0.06,0.00,-0.07,0.01,0.00,0.0,0.00,0.0,0.0,0.0
1970-01-01 00:29:40.199666616,BTCUSDT,74165.66,74165.67,74165.665,74165.660605,7416566,7416567,7416567,0.01,-0.875744,...,-0.06,0.00,-0.07,0.01,0.00,0.0,0.00,0.0,0.0,0.0
1970-01-01 00:29:40.199666714,BTCUSDT,74165.66,74165.67,74165.665,74165.660622,7416566,7416567,7416567,0.01,-0.872296,...,-0.06,0.00,-0.07,0.01,0.00,0.0,0.00,0.0,0.0,0.0
1970-01-01 00:29:40.199666814,BTCUSDT,74165.66,74165.67,74165.665,74165.660622,7416566,7416567,7416567,0.01,-0.872296,...,-0.06,0.00,-0.07,0.01,0.00,0.0,0.00,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1970-01-01 00:29:40.215916114,BTCUSDT,73975.62,73975.63,73975.625,73975.629575,7397562,7397563,7397562,0.01,0.918478,...,0.00,0.08,-0.01,0.09,0.01,0.0,0.01,0.0,0.0,0.0
1970-01-01 00:29:40.215916214,BTCUSDT,73975.62,73975.63,73975.625,73975.629155,7397562,7397563,7397562,0.01,0.834382,...,0.00,0.08,-0.01,0.09,0.01,0.0,0.01,0.0,0.0,0.0
1970-01-01 00:29:40.215916314,BTCUSDT,73975.62,73975.63,73975.625,73975.629194,7397562,7397563,7397562,0.01,0.842143,...,0.00,0.08,-0.01,0.09,0.01,0.0,0.01,0.0,0.0,0.0


In [11]:
# STEP 5 — ALIGN BOTH DATASETS

# This is the missing step in your code.

# Now regime + outcome are aligned.

final = regime_df.join(
    eval_df[[
        "future_return",
        "future_volatility",
        "future_direction",
        "mid",
        "microprice"
    ]],
    how="inner"
)

In [12]:
# STEP 6 — ANALYZE REGIMES

final.groupby("regime").agg({
    "future_return": "mean",
    "spread": "mean",
    "microprice_error": "mean",
    "future_direction": "mean"
})

# Research phase

# There is some human interpretation.

# You might see:

# regime	future_return	future_volatility
# 0	0.0000	0.0004
# 1	0.0001	0.0030
# 2	0.0015	0.0005

# and conclude:

# 0 = STABLE_MM
# 1 = HIGH_VOL
# 2 = TRENDING

# high future return → trending regime
# high future volatility → stress regime
# high microprice error → inefficient regime (alpha-rich)
# low everything → stable MM regime

# That's "eyeballing", but it's not arbitrary. You're looking at measurable statistics.

,future_return,spread,microprice_error,future_direction
regime,,,,
0,-2.196785e-07,0.01006,0.000367,-0.010892
1,1.519865e-07,0.01000,0.000397,0.002462
2,-5.740129e-07,0.01000,0.001730,-0.016591
3,-6.382630e-08,0.01003,0.000544,-0.013336


In [ ]:
feature_cols = [
    "volatility",
    "spread",
    "order_imbalance",
    "trade_imbalance",
    "quote_churn",
    "inventory",
    "inventory_vol",
    "microprice_error",
]

z = final.copy()

for col in feature_cols:
    z[col] = (
        z[col] - z[col].mean()
    ) / z[col].std()

regime_profile = (
    z.groupby("regime")[feature_cols]
    .mean()
    .round(2)
)

print(regime_profile)

"""
At that point the labels become obvious:

highest volatility + churn → High Vol
highest microprice error + imbalance + future drift → Toxic
lowest volatility/churn → Low Vol
strongest directional imbalance but moderate vol → Trending

Normal low-vol regime

Suppose:

Bid: 100.00 (100 BTC)
Ask: 100.01 (100 BTC)

Mid = 100.005
Microprice ≈ 100.005

Book is balanced.

Nothing special.

Potential informed-flow regime

Now imagine:

Bid: 100.00 (5 BTC)
Ask: 100.01 (200 BTC)

Mid = 100.005
Microprice = 100.000x

The displayed mid hasn't moved.

But the book is screaming:

there is almost no buying interest
there is massive sell-side pressure

Microprice moves far below mid.

That's exactly what your regime 2 seems to be detecting:

microprice_error = +4.56 z-score

which is enormous.

Why low volatility actually strengthens the case

Suppose volatility were huge.

Then microprice error could simply be noise:

book moving everywhere
quotes flickering
price jumping around

Hard to extract information.

But your cluster is:

volatility      -3.74
quote_churn     -3.69

Meaning:

book is stable
market is calm
yet imbalance is extreme

That's much more suspicious.

You essentially have:

Price says: nothing happening

Book says: something is about to happen

That's exactly the situation a market maker fears.
"""

        volatility  spread  order_imbalance  trade_imbalance  quote_churn  \
regime                                                                      
0             0.46    0.61             0.28             0.38         0.40   
1            -1.81   -1.95             0.16            -0.36        -1.86   
2            -3.74   -1.97            -4.54            -2.95        -3.69   
3            -0.05   -0.68            -0.39            -0.63         0.16   

        inventory  inventory_vol  microprice_error  
regime                                              
0            0.63           0.59             -0.27  
1           -1.56          -2.00             -0.17  
2           -1.93          -2.97              4.56  
3           -0.94          -0.48              0.35  


In [14]:
# 5. Your detect_regime() function is fine BUT incomplete

# You MUST ensure:

# same feature order
# same scaler trained on regime_df
# no missing values

def detect_regime(self, features):
    x = np.array([
        features["volatility"],
        features["spread"],
        features["order_imbalance"],
        features["trade_imbalance"],
        features["quote_churn"],
        features["inventory"],
        features["inventory_vol"],
        features["microprice_error"],
    ]).reshape(1, -1)

    x = self.scaler.transform(x)
    return self.regime_model.predict(x)[0]

In [31]:
feature_cols = [
    "volatility",
    "spread",
    "order_imbalance",
    "trade_imbalance",
    "quote_churn",
    "inventory",
    "inventory_vol",
    "microprice_error",
]

x = regime_df.iloc[[0]][feature_cols]

x_scaled = scaler.transform(x)

pred = model.predict(x_scaled)[0]

probs = model.predict_proba(x_scaled)[0]

print("regime probability:", probs)

print("prediction:", pred)

regime probability: [0. 0. 1. 0.]
prediction: 2


In [32]:
import joblib

regime_labels = {
    0: "high_vol",
    1: "low_vol",
    2: "toxic",
    3: "neutral",
}

artifact = {
    "scaler": scaler,
    "model": model,
    "feature_cols": feature_cols,
    "n_regimes": n_regimes,
    "window": WINDOW,
    "regime_labels": regime_labels
}

joblib.dump(artifact, "regime_model.pkl")

['regime_model.pkl']

In [ ]:
"""
Key Research Questions

This project is designed to investigate:

When does microprice provide predictive value?
Which market states produce adverse selection?
How does queue position impact fill quality?
Which regimes favor passive liquidity provision?
How should inventory management adapt to changing conditions?
How much PnL comes from spread capture versus directional edge?
Key Insights

Market making profitability emerges from the interaction of:

alpha quality
execution quality
inventory control
market regime

No individual component is sufficient in isolation.

Execution without alpha becomes adverse selection.

Alpha without execution becomes unrealized opportunity.

Inventory control without regime awareness can dominate spread capture gains.

Core Takeaway

Profitable market making is an adaptive liquidity provision problem where alpha forecasting, execution quality, inventory management, and market regime jointly determine realized PnL.
"""